# Inference 4 LLaMA konfigurací s dlouhým promptem

Inference všech 4 lokálních LLaMA konfigurací na 80 testovacích záznamech s dlouhou variantou systémového promptu.

Výstup: `outputs/predictions_long/`

## 1. Vypnutí Unsloth telemetrie + instalace

In [1]:
import os
os.environ['UNSLOTH_DISABLE_STATISTICS'] = '1'
os.environ['HF_HUB_OFFLINE'] = '0'
print('Unsloth telemetry vypnuta')

✅ Unsloth telemetry vypnuta


In [2]:
%%capture
!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps peft accelerate bitsandbytes
!pip install transformers datasets

## 2. Připojení Drive + cesty

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_ROOT = '/content/drive/MyDrive/bp-misleading-cs'

TEST_DATA_PATH = f'{PROJECT_ROOT}/data/processed/test.jsonl'
SYSTEM_PROMPT_PATH = f'{PROJECT_ROOT}/prompts/system_prompt_v1.txt'
BASE_ADAPTER = f'{PROJECT_ROOT}/models/llama_base_qlora_v1'
INSTRUCT_ADAPTER = f'{PROJECT_ROOT}/models/llama_instruct_qlora_v1'
OUTPUT_DIR = f'{PROJECT_ROOT}/outputs/predictions_long'

os.makedirs(OUTPUT_DIR, exist_ok=True)

for path in [TEST_DATA_PATH, SYSTEM_PROMPT_PATH, BASE_ADAPTER, INSTRUCT_ADAPTER]:
    assert os.path.exists(path), f'Chybí: {path}'
    print(f'{path}')
print(f'\noutput: {OUTPUT_DIR}')

✅ /content/drive/MyDrive/bp-misleading-cs/data/processed/test.jsonl
✅ /content/drive/MyDrive/bp-misleading-cs/prompts/system_prompt_v1.txt
✅ /content/drive/MyDrive/bp-misleading-cs/models/llama_base_qlora_v1
✅ /content/drive/MyDrive/bp-misleading-cs/models/llama_instruct_qlora_v1

✅ output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions


## 3. HuggingFace login

In [5]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('Přihlášeno do HuggingFace')

✅ Přihlášeno do HuggingFace


## 4. Načtení test dat + system promptu

In [6]:
import json

# Načíst system prompt
with open(SYSTEM_PROMPT_PATH, 'r', encoding='utf-8') as f:
    SYSTEM_PROMPT = f.read().strip()

print(f'System prompt: {len(SYSTEM_PROMPT)} znaků')
print(f'Prvních 200 znaků: {SYSTEM_PROMPT[:200]}...')

# Načíst test set
test_records = []
with open(TEST_DATA_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        test_records.append(json.loads(line.strip()))

print(f'\nTest set: {len(test_records)} záznamů')
print(f'\nUkázka prvního záznamu:')
print(f'  id: {test_records[0]["id"]}')
print(f'  header: {test_records[0]["header"]}')
print(f'  E1: {test_records[0]["annotation"]["E1-misleading_header_model_final"]}')

System prompt: 6419 znaků
Prvních 200 znaků: Jsi expert na analýzu zpravodajských sdělení. Tvým úkolem je vyhodnotit česky psaný titulek a posoudit jeho zavádějícnost podle definovaného schématu.

VSTUPNÍ JEDNOTKA
Vstupem je samostatný titulek. ...

Test set: 80 záznamů

Ukázka prvního záznamu:
  id: reflex_24eac431
  header: Tahá prezident za kratší konec Ústavy než Macinka? Divoká kohabitace poškozuje zájmy Česka
  E1: POTENTIALLY_MISLEADING


## 5. Helper funkce

In [7]:
import re
from datetime import datetime

ALPACA_TEMPLATE = '''### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}'''


def parse_json_output(raw_output: str) -> tuple:
    """
    Robustní parser. Pokouší se vyextrahovat validní JSON z výstupu modelu.

    Returns: (parsed_dict_or_None, status_string)
    """
    if not raw_output or not raw_output.strip():
        return None, 'empty_output'

    text = raw_output.strip()

    # Strip markdown code fences
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```\s*$', '', text)

    # Pokus 1: Přímý parse
    try:
        return json.loads(text), 'ok'
    except json.JSONDecodeError:
        pass

    # Pokus 2: Najdi první { a poslední } a parse mezi nimi
    first_brace = text.find('{')
    last_brace = text.rfind('}')
    if first_brace >= 0 and last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate), 'ok_extracted'
        except json.JSONDecodeError:
            pass

    # Pokus 3: Greedy regex pro největší {...} blok
    matches = re.findall(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', text)
    for match in matches:
        try:
            return json.loads(match), 'ok_regex'
        except json.JSONDecodeError:
            continue

    return None, 'failed'


def run_inference(model, tokenizer, test_records, config_name, output_dir):
    """
    Spustí inferenci na všech test záznamech.
    Průběžně ukládá výstupy do JSONL souboru.
    """
    output_path = os.path.join(output_dir, f'{config_name}.jsonl')

    print(f'\n{"="*60}')
    print(f'KONFIGURACE: {config_name}')
    print(f'Output: {output_path}')
    print(f'Záznamů: {len(test_records)}')
    print(f'{"="*60}')

    parsing_stats = {'ok': 0, 'ok_extracted': 0, 'ok_regex': 0, 'failed': 0, 'empty_output': 0}
    start_time = datetime.now()

    with open(output_path, 'w', encoding='utf-8') as f_out:
        for idx, record in enumerate(test_records, 1):
            # Sestav prompt
            prompt = ALPACA_TEMPLATE.format(
                instruction=SYSTEM_PROMPT,
                input=f'Titulek: "{record["header"]}"',
                output='',
            )

            # Inference
            inputs = tokenizer([prompt], return_tensors='pt').to('cuda')
            outputs = model.generate(
                **inputs,
                max_new_tokens=512,
                temperature=0.0,
                do_sample=False,
                use_cache=True,
                pad_token_id=tokenizer.eos_token_id,
            )

            # Vytáhni jen response part
            full_output = tokenizer.batch_decode(outputs)[0]
            try:
                response = full_output.split('### Response:\n')[1].split(tokenizer.eos_token)[0].strip()
            except IndexError:
                response = full_output

            # Parse JSON
            parsed, status = parse_json_output(response)
            parsing_stats[status] = parsing_stats.get(status, 0) + 1

            # Zapiš do JSONL
            output_record = {
                'id': record['id'],
                'header': record['header'],
                'config': config_name,
                'raw_output': response,
                'parsed_output': parsed,
                'parsing_status': status,
                'gold_annotation': record['annotation'],
            }
            f_out.write(json.dumps(output_record, ensure_ascii=False) + '\n')
            f_out.flush()  # průběžné ukládání

            # Progress every 10 záznamů
            if idx % 10 == 0:
                elapsed = (datetime.now() - start_time).total_seconds()
                eta = elapsed / idx * (len(test_records) - idx)
                print(f'  [{idx}/{len(test_records)}] elapsed: {elapsed:.0f}s, ETA: {eta:.0f}s')

    # Souhrn
    elapsed_total = (datetime.now() - start_time).total_seconds()
    print(f'\n{config_name} hotovo za {elapsed_total:.0f}s')
    print(f'Parsing statistika:')
    for status, count in parsing_stats.items():
        if count > 0:
            pct = count / len(test_records) * 100
            print(f'  {status}: {count} ({pct:.1f}%)')

    return parsing_stats


print('Helper funkce definované')

✅ Helper funkce definované


## 6. KONFIGURACE 1: `llama_base_zs`

LLaMA 3.1 8B Base, bez adapteru (baseline).

In [11]:
from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Meta-Llama-3.1-8B',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)
FastLanguageModel.for_inference(model)

print(f'Model načten: {model.config._name_or_path}')

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model načten: unsloth/meta-llama-3.1-8b-unsloth-bnb-4bit


In [12]:
stats_base_zs = run_inference(model, tokenizer, test_records, 'llama_base_zs_long', OUTPUT_DIR)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



KONFIGURACE: llama_base_zs
Output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions/llama_base_zs.jsonl
Záznamů: 80


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=

  [10/80] elapsed: 84s, ETA: 588s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [60/80] elapsed: 506s, ETA: 169s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [70/80] elapsed: 590s, ETA: 84s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [80/80] elapsed: 690s, ETA: 0s

✅ llama_base_zs hotovo za 690s
Parsing statistika:
  ok: 80 (100.0%)


## 7. KONFIGURACE 2: `llama_base_qlora_zs`

LLaMA 3.1 8B Base + QLoRA adapter.

In [13]:
from peft import PeftModel

try:
    model = model.unload()
    print('Předchozí adapter unloaded')
except Exception:
    pass

model = PeftModel.from_pretrained(model, BASE_ADAPTER)
FastLanguageModel.for_inference(model)

print(f'Adapter aplikován: {BASE_ADAPTER}')

✅ Adapter aplikován: /content/drive/MyDrive/bp-misleading-cs/models/llama_base_qlora_v1


In [14]:
stats_base_qlora = run_inference(model, tokenizer, test_records, 'llama_base_qlora_zs_long', OUTPUT_DIR)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



KONFIGURACE: llama_base_qlora_zs
Output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions/llama_base_qlora_zs.jsonl
Záznamů: 80


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [10/80] elapsed: 94s, ETA: 658s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [60/80] elapsed: 572s, ETA: 191s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [70/80] elapsed: 661s, ETA: 94s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [80/80] elapsed: 763s, ETA: 0s

✅ llama_base_qlora_zs hotovo za 763s
Parsing statistika:
  ok: 80 (100.0%)


## 8. Uvolnění paměti pro Instruct

In [15]:
import gc

del model
del tokenizer
gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

# Ověřit volnou paměť
free_gb = torch.cuda.mem_get_info()[0] / 1e9
total_gb = torch.cuda.mem_get_info()[1] / 1e9
print(f'GPU paměť: {free_gb:.1f} / {total_gb:.1f} GB volné')

GPU paměť: 16.6 / 23.7 GB volné


## 9. KONFIGURACE 3: `llama_instruct_zs`

LLaMA 3.1 8B Instruct, bez adapteru (baseline).

In [16]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='meta-llama/Meta-Llama-3.1-8B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=hf_token,
)
FastLanguageModel.for_inference(model)

print(f'Model načten: {model.config._name_or_path}')

==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


✅ Model načten: unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit


In [17]:
stats_instruct_zs = run_inference(model, tokenizer, test_records, 'llama_instruct_zs_long', OUTPUT_DIR)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



KONFIGURACE: llama_instruct_zs
Output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions/llama_instruct_zs.jsonl
Záznamů: 80


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [10/80] elapsed: 235s, ETA: 1644s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [60/80] elapsed: 1407s, ETA: 469s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [70/80] elapsed: 1642s, ETA: 235s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [80/80] elapsed: 1876s, ETA: 0s

✅ llama_instruct_zs hotovo za 1876s
Parsing statistika:
  ok_extracted: 80 (100.0%)


## 10. KONFIGURACE 4: `llama_instruct_qlora_zs`

LLaMA 3.1 8B Instruct + QLoRA adapter.

In [18]:
try:
    model = model.unload()
    print('Předchozí adapter unloaded')
except Exception:
    pass

model = PeftModel.from_pretrained(model, INSTRUCT_ADAPTER)
FastLanguageModel.for_inference(model)

print(f'Adapter aplikován: {INSTRUCT_ADAPTER}')

✅ Adapter aplikován: /content/drive/MyDrive/bp-misleading-cs/models/llama_instruct_qlora_v1


In [19]:
stats_instruct_qlora = run_inference(model, tokenizer, test_records, 'llama_instruct_qlora_zs_long', OUTPUT_DIR)

Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



KONFIGURACE: llama_instruct_qlora_zs
Output: /content/drive/MyDrive/bp-misleading-cs/outputs/predictions/llama_instruct_qlora_zs.jsonl
Záznamů: 80


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [10/80] elapsed: 93s, ETA: 649s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

...

  [60/80] elapsed: 563s, ETA: 188s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [70/80] elapsed: 652s, ETA: 93s


Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_

  [80/80] elapsed: 752s, ETA: 0s

✅ llama_instruct_qlora_zs hotovo za 752s
Parsing statistika:
  ok: 80 (100.0%)


## 11. Souhrn všech 4 konfigurací

In [20]:
print('='*70)
print('SOUHRN PARSING SUCCESS RATE')
print('='*70)

all_stats = {
    'llama_base_zs_long': stats_base_zs,
    'llama_base_qlora_zs_long': stats_base_qlora,
    'llama_instruct_zs_long': stats_instruct_zs,
    'llama_instruct_qlora_zs_long': stats_instruct_qlora,
}

print(f'\n{"Konfigurace":<28} {"OK":>8} {"OK*":>8} {"Failed":>8} {"Empty":>8} {"Total OK %":>12}')
print('-' * 72)
for name, stats in all_stats.items():
    ok = stats.get('ok', 0)
    ok_extracted = stats.get('ok_extracted', 0) + stats.get('ok_regex', 0)
    failed = stats.get('failed', 0)
    empty = stats.get('empty_output', 0)
    total = ok + ok_extracted + failed + empty
    total_ok_pct = (ok + ok_extracted) / total * 100 if total > 0 else 0
    print(f'{name:<28} {ok:>8} {ok_extracted:>8} {failed:>8} {empty:>8} {total_ok_pct:>11.1f}%')

print('\nLegenda:')
print('  OK     = parsing prošel přímo')
print('  OK*    = parsing po extrakci/regex')
print('  Failed = nepodařilo se vyextrahovat JSON')
print('  Empty  = model nevygeneroval žádný výstup')

SOUHRN PARSING SUCCESS RATE

Konfigurace                        OK      OK*   Failed    Empty   Total OK %
------------------------------------------------------------------------
llama_base_zs                      80        0        0        0       100.0%
llama_base_qlora_zs                80        0        0        0       100.0%
llama_instruct_zs                   0       80        0        0       100.0%
llama_instruct_qlora_zs            80        0        0        0       100.0%

Legenda:
  OK     = parsing prošel přímo
  OK*    = parsing po extrakci/regex
  Failed = nepodařilo se vyextrahovat JSON
  Empty  = model nevygeneroval žádný výstup
